# Риманова модель с обучаемым эмбеддингом участника

1. **Ковариации** окон (4–40 Гц).
2. **Причинное перецентрирование сессии**: опорная точка сессии — бегущее лог-евклидово среднее её ковариаций, без меток. Окно отображаем в касательное пространство в этой точке.
3. **Эмбеддинг участника.** Перецентрирование убирает индивидуальность человека, но её можно вернуть модели как условие:
   - *отпечаток* сессии = сдвиг её опорной точки от общего среднего;
   - эмбеддинг `e = encoder(отпечаток) + table[участник]`; FiLM-слой подстраивает классификатор под `e`.
   - Таблица `table` обучаемая, но есть только для train-участников. Поэтому при обучении она случайно выключается (subject dropout), и encoder учится восстанавливать эмбеддинг по одному отпечатку. На тесте все участники новые, и работает только encoder.

Оценка как на платформе: окна идут по одному, каждая сессия начинается с чистой копии модели.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.signal import butter, sosfiltfilt
from pyriemann.estimation import Covariances
from pyriemann.utils.base import logm, expm, invsqrtm
from pyriemann.utils.tangentspace import upper
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from bci.data import load_split, SFREQ
from bci.evaluation import evaluate, score, compare_with_baselines

torch.manual_seed(0); np.random.seed(0)
DATA_DIR = "../data"
train, val, test = (load_split(s, DATA_DIR) for s in ["train", "val", "test"])

## Признаки: ковариации и причинное перецентрирование

In [ ]:
SOS = butter(4, [4, 40], btype="bandpass", fs=SFREQ, output="sos")

def covariances(X):
    """(N, C, T) -> (N, C, C)"""
    return Covariances("oas").transform(sosfiltfilt(SOS, np.asarray(X, np.float64), axis=-1))


class SessionRecentering:
    """Опорная точка сессии = бегущее лог-евклидово среднее её ковариаций.
    Стартует с общего среднего train (prior) с весом prior_weight окон."""

    def __init__(self, log_prior, prior_weight=5):
        self.log_prior, self.prior_weight = log_prior, prior_weight
        self.reset()

    def reset(self):
        self.log_sum, self.n = self.prior_weight * self.log_prior, self.prior_weight

    def __call__(self, cov):
        """-> (признаки окна, отпечаток сессии)"""
        self.log_sum = self.log_sum + logm(cov)
        self.n += 1
        log_ref = self.log_sum / self.n
        W = invsqrtm(expm(log_ref))
        return upper(logm(W @ cov @ W)), upper(log_ref - self.log_prior)


def session_features(windows, covs, recenter):
    """Признаки всех окон, посчитанные так же причинно, как на тесте."""
    x, fp = [None] * len(windows), [None] * len(windows)
    for idx in windows.streams():
        recenter.reset()
        for i in idx:
            x[i], fp[i] = recenter(covs[i])
    return np.array(x), np.array(fp)


train_covs, val_covs = covariances(train.X), covariances(val.X)
recenter = SessionRecentering(log_prior=np.mean([logm(c) for c in train_covs], axis=0))
x_tr, fp_tr = session_features(train, train_covs, recenter)
x_va, fp_va = session_features(val, val_covs, recenter)
x_scaler, fp_scaler = StandardScaler().fit(x_tr), StandardScaler().fit(fp_tr)
print("признаки:", x_tr.shape, " отпечатки:", fp_tr.shape)

## Модель, которую сдаём: обёртка под контракт платформы

In [ ]:
class StreamingDecoder:
    """Контракт Track 2: reset_state() и predict(X) для X формы (1, C, T)."""

    def __init__(self, classify):
        self.classify = classify  # (x, fp) -> класс
        self.recenter = SessionRecentering(recenter.log_prior)

    def reset_state(self):
        self.recenter.reset()

    def predict(self, X):
        x, fp = self.recenter(covariances(X)[0])
        x, fp = x_scaler.transform(x[None]), fp_scaler.transform(fp[None])
        return np.array([self.classify(x, fp)])

## Простой ориентир: перецентрирование + логистическая регрессия

In [ ]:
logreg = LogisticRegression(C=0.01, max_iter=2000, class_weight="balanced")
logreg.fit(x_scaler.transform(x_tr), train.y)
print("val:", score(val, logreg.predict(x_scaler.transform(x_va))))

## Эмбеддинг участника + FiLM

In [ ]:
class SubjectFiLM(nn.Module):
    def __init__(self, n_feat, n_subjects, n_classes=3, dim=32, emb=8, subject_dropout=0.5):
        super().__init__()
        self.proj = nn.Linear(n_feat, dim)
        self.encoder = nn.Linear(n_feat, emb)        # отпечаток сессии -> эмбеддинг
        self.table = nn.Embedding(n_subjects, emb)   # обучаемая добавка для train-участников
        self.film = nn.Linear(emb, 2 * dim)
        self.head = nn.Linear(dim, n_classes)
        self.drop_input, self.subject_dropout = nn.Dropout(0.3), subject_dropout

    def embed(self, fp, subject=None):
        e = torch.tanh(self.encoder(fp))
        if subject is not None:
            keep = (torch.rand(len(fp), 1) > self.subject_dropout).float()
            e = e + keep * self.table(subject)
        return e

    def forward(self, x, fp, subject=None):
        gamma, beta = self.film(self.embed(fp, subject)).chunk(2, dim=-1)
        h = torch.relu(self.proj(self.drop_input(x)) * (1 + gamma) + beta)
        return self.head(h)

In [ ]:
T = lambda a: torch.tensor(a, dtype=torch.float32)
subjects = sorted(set(train.subject))
X_tr, F_tr, Y_tr = T(x_scaler.transform(x_tr)), T(fp_scaler.transform(fp_tr)), torch.tensor(train.y)
S_tr = torch.tensor([subjects.index(s) for s in train.subject])
X_va, F_va = T(x_scaler.transform(x_va)), T(fp_scaler.transform(fp_va))

net = SubjectFiLM(X_tr.shape[1], len(subjects))
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
class_weights = T(len(Y_tr) / (3 * np.bincount(train.y, minlength=3)))
loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

best, history = (-1, None), []
for epoch in range(300):
    net.train(); opt.zero_grad()
    loss = loss_fn(net(X_tr, F_tr, S_tr), Y_tr)
    loss.backward(); opt.step()

    net.eval()
    with torch.no_grad():
        val_score = score(val, net(X_va, F_va).argmax(-1).numpy())["session_mean"]
    history.append(val_score)
    if val_score > best[0]:
        best = (val_score, {k: v.clone() for k, v in net.state_dict().items()})

net.load_state_dict(best[1]); net.eval()
print(f"лучший val session_mean: {best[0]:.3f} (эпоха {int(np.argmax(history))})")
pd.Series(history).plot(title="val session_mean по эпохам");

## Честная оценка на test: поток окон, как на платформе

In [ ]:
def film_classify(x, fp):
    with torch.no_grad():
        return int(net(T(x), T(fp)).argmax(-1))

results = {
    "Riemann + LogReg": evaluate(StreamingDecoder(lambda x, fp: int(logreg.predict(x)[0])), test),
    "Riemann + subject FiLM": evaluate(StreamingDecoder(film_classify), test),
}
compare_with_baselines(results)

Эмбеддинги сессий: encoder видит только неразмеченные ковариации, а сессии одного человека всё равно оказываются рядом.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

with torch.no_grad():
    E = net.embed(F_tr).numpy()
z = PCA(2).fit_transform(E)
plt.scatter(*z.T, c=pd.factorize(train.subject)[0], cmap="tab10", s=4)
plt.title("эмбеддинг из отпечатка, train (цвет = участник)");